# LangChain Messages

এই notebook-এ LangChain-এর বিভিন্ন ধরনের **message** দেখানো হবে।

LangChain-এ LLM-এর সাথে কথা বলতে হলে message পাঠাতে হয়। বিভিন্ন কাজের জন্য বিভিন্ন ধরনের message ব্যবহার করা হয়।

| Message Type | কে পাঠায় | কাজ কী |
|---|---|---|
| Text Prompt | User | সরাসরি string দিয়ে invoke করা |
| SystemMessage | Developer | AI-কে তার role বলে দেওয়া |
| HumanMessage | User | মানুষের প্রশ্ন বা কথা |
| AIMessage | LLM | AI-এর উত্তর (tool call সহ বা ছাড়া) |
| ToolMessage | Tool | tool-এর result যা AI পড়বে |

**সব example-এ একই `weather_db` ব্যবহার করা হয়েছে।**

### Step 1 — Environment Setup

**কী হচ্ছে:** `.env` file থেকে `ANTHROPIC_API_KEY` load করা হচ্ছে।

**কেন:** API key কখনো code-এ লেখা যাবে না। `python-dotenv` দিয়ে `.env` file থেকে পড়লে key টা safe থাকে।

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
print("ANTHROPIC_API_KEY:", bool(ANTHROPIC_API_KEY))

ANTHROPIC_API_KEY: True


---
### Step 2 — Weather Tool এবং LLM তৈরি

**কী হচ্ছে:** আগের notebook-এর মতোই `get_weather` tool এবং `ChatAnthropic` LLM তৈরি করা হচ্ছে।

**কেন এখানে আবার দেওয়া হলো:** Message type-গুলো বোঝাতে গেলে একটা real tool দরকার যেটার result `ToolMessage`-এ দেখানো যাবে।

In [2]:
from langchain_core.tools import tool
from langchain_anthropic import ChatAnthropic

@tool
def get_weather(city: str) -> str:
    """Return the current weather for a given city name."""
    weather_db = {
        "dhaka":      {"temperature": "34°C", "condition": "Sunny",         "humidity": "72%", "wind_speed": "10 km/h", "feels_like": "38°C"},
        "chittagong": {"temperature": "32°C", "condition": "Partly Cloudy", "humidity": "78%", "wind_speed": "14 km/h", "feels_like": "36°C"},
        "london":     {"temperature": "17°C", "condition": "Overcast",      "humidity": "85%", "wind_speed": "20 km/h", "feels_like": "15°C"},
    }
    key = city.lower().strip()
    if key not in weather_db:
        return f"No weather data available for '{city}'."
    w = weather_db[key]
    return (
        f"Weather in {city.title()}:\n"
        f"  Temperature : {w['temperature']} (feels like {w['feels_like']})\n"
        f"  Condition   : {w['condition']}\n"
        f"  Humidity    : {w['humidity']}\n"
        f"  Wind Speed  : {w['wind_speed']}"
    )

llm = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
    anthropic_api_key=ANTHROPIC_API_KEY,
)

print("LLM and tool ready.")

LLM and tool ready.


---
## 1. Text Prompt (সরাসরি string)

**কী হচ্ছে:** `.invoke()` তে সরাসরি একটা plain string দেওয়া হচ্ছে।

**কীভাবে কাজ করে:** LangChain নিজে থেকে string টাকে `HumanMessage`-এ মুড়ে দেয়। তাই আলাদা করে message object বানাতে হয় না।

**কখন ব্যবহার করবে:** একটাই simple প্রশ্ন করলে এটাই সবচেয়ে সহজ।

In [3]:
# Plain string — LangChain internally wraps this as a HumanMessage
response = llm.invoke("Dhaka-র আজকের আবহাওয়া কেমন?")

print("Type  :", type(response).__name__)
print("Answer:", response.content)

Type  : AIMessage
Answer: আমার কাছে রিয়েল-টাইম তথ্য নেই, তাই আমি আজকের আবহাওয়া সরাসরি বলতে পারছি না।

ঢাকার বর্তমান আবহাওয়া জানতে আপনি এই উপায়গুলো ব্যবহার করতে পারেন:

1. **অনলাইন সার্চ**: Google বা আবহাওয়া ওয়েবসাইটে "Dhaka weather today" লিখুন
2. **আবহাওয়া অ্যাপ**: 
   - Weather.com
   - AccuWeather
   - বাংলাদেশ আবহাওয়া অধিদপ্তর (BMD) এর ওয়েবসাইট
3. **মোবাইল অ্যাপ**: আপনার ফোনের built-in weather অ্যাপ

এই সব উৎস থেকে তাপমাত্রা, আর্দ্রতা, বৃষ্টির সম্ভাবনা ইত্যাদি সম্পর্কে সঠিক তথ্য পাবেন।

আর কোনো সাহায্য দরকার?


---
## 2. SystemMessage

**কী হচ্ছে:** `SystemMessage` দিয়ে AI-কে তার **role বা behavior** বলে দেওয়া হচ্ছে।

**কীভাবে কাজ করে:** System message সবার আগে আসে এবং AI-এর পুরো conversation-এর tone ও character ঠিক করে দেয়। User এটা দেখতে পায় না।

**কখন ব্যবহার করবে:** AI-কে বলতে চাইলে — "তুমি একজন weather expert", "তুমি শুধু বাংলায় উত্তর দেবে", "তুমি ছোট উত্তর দেবে" ইত্যাদি।

In [4]:
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content="তুমি একজন বাংলাদেশী আবহাওয়া বিশেষজ্ঞ। সব উত্তর সহজ বাংলায় দেবে।"),
    HumanMessage(content="ঢাকার আবহাওয়া কেমন হতে পারে আজকে?"),
]

response = llm.invoke(messages)

print("SystemMessage effect →")
print(response.content)

SystemMessage effect →
আমি আপনাকে সঠিক তথ্য দিতে পারছি না কারণ আমার কাছে আজকের সঠিক আবহাওয়া তথ্য নেই। আমার ডেটা সীমিত এবং রিয়েল-টাইম আপডেট পাই না।

**আজকের আবহাওয়া জানার জন্য আপনি:**

১. **বাংলাদেশ আবহাওয়া অধিদপ্তরের ওয়েবসাইট** দেখতে পারেন - www.bmd.gov.bd

২. **আবহাওয়া অ্যাপ** ব্যবহার করতে পারেন - যেমন Google Weather বা AccuWeather

৩. **টিভি সংবাদ** থেকে আবহাওয়া বুলেটিন শুনতে পারেন

৪. **সোশ্যাল মিডিয়া** - বাংলাদেশ আবহাওয়া অধিদপ্তরের ফেসবুক পেজ ফলো করতে পারেন

যদি আপনি সাধারণ আবহাওয়া প্যাটার্ন বা ঋতু সম্পর্কে জানতে চান, তাহলে আমি সাহায্য করতে পারব। আপনার আর কোনো প্রশ্ন আছে?


---
## 3. HumanMessage

**কী হচ্ছে:** `HumanMessage` হলো user বা মানুষের পক্ষ থেকে আসা message।

**কীভাবে কাজ করে:** এটা conversation history-তে user-এর turn হিসেবে যোগ হয়। Multi-turn conversation-এ প্রতিটি user input `HumanMessage` হিসেবে রাখা হয়।

**Text Prompt থেকে পার্থক্য:** Text Prompt একটা shortcut — `HumanMessage` দিলে explicit এবং অন্য message-এর সাথে মিশিয়ে list আকারে পাঠানো যায়।

In [5]:
from langchain_core.messages import HumanMessage

# Multi-turn conversation: দুইটা HumanMessage পাঠানো
messages = [
    HumanMessage(content="লন্ডন আর চট্টগ্রামের মধ্যে কোনটায় এখন বেশি গরম?"),
]

response = llm.invoke(messages)

print("HumanMessage response →")
print(response.content)

HumanMessage response →
এটা নির্ভর করে বর্তমান মৌসুমের উপর:

**সাধারণত:**
- **চট্টগ্রাম** গরম ও আর্দ্র (বছরের বেশিরভাগ সময় ২৫-৩৫°সেলসিয়াস)
- **লন্ডন** মৃদু জলবায়ু (গ্রীষ্মে ১৫-২৫°সেলসিয়াস, শীতে ০-৫°সেলসিয়াস)

**এখন (ডিসেম্বর):**
- **চট্টগ্রাম**: শীতকাল শুরু হয়েছে, তবুও ২০-২৫°সেলসিয়াস
- **লন্ডন**: শীত, ৫-১০°সেলসিয়াস বা কম

তাই **এখন চট্টগ্রামে বেশি গরম** আছে।

আপনি যদি সঠিক উত্তর চান, আমাকে আজকের তারিখ বা নির্দিষ্ট সময় জানাতে পারেন।


---
## 4. AIMessage

**কী হচ্ছে:** `AIMessage` হলো LLM-এর দেওয়া উত্তর। `.invoke()` সবসময় `AIMessage` return করে।

**দুই ধরনের AIMessage আছে:**
- **Normal reply** — `.content` তে text থাকে, `.tool_calls` খালি।
- **Tool call reply** — `.content` খালি বা ছোট, `.tool_calls` এ tool-এর নাম ও argument থাকে।

**কেন জানা দরকার:** Multi-turn conversation build করতে হলে AI-এর আগের reply `AIMessage` হিসেবে history-তে রাখতে হয়, তাহলে পরের turn-এ AI আগের কথা মনে রাখে।

In [6]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# ── Normal AIMessage ──
response = llm.invoke("ঢাকার গরমে কী পোশাক পরা ভালো?")

print("=== Normal AIMessage ===")
print("Type        :", type(response).__name__)
print("Content     :", response.content)
print("Tool calls  :", response.tool_calls)

# ── AIMessage with tool_call (bind a tool first) ──
llm_with_tools = llm.bind_tools([get_weather])
tc_response = llm_with_tools.invoke("ঢাকার আবহাওয়া কী?")

print("\n=== AIMessage with Tool Call ===")
print("Type        :", type(tc_response).__name__)
print("Content     :", repr(tc_response.content))
print("Tool calls  :", tc_response.tool_calls)

=== Normal AIMessage ===
Type        : AIMessage
Content     : # ঢাকার গরমে পোশাকের পরামর্শ

## সবচেয়ে ভালো বিকল্পগুলো:

**ফ্যাব্রিক:**
- **সুতি (কটন)** - শ্বাসপ্রশ্বাসযোগ্য এবং আরামদায়ক
- **লিনেন** - হালকা এবং ঘাম শোষণ করে
- **মসলিন** - ঐতিহ্যবাহী এবং আরামদায়ক

**পোশাকের ধরন:**
- আলগা এবং ঢিলেঢালা পোশাক
- ছোট হাতের শার্ট বা টপস
- হালকা রঙের পোশাক (সাদা, হালকা নীল, পেস্টেল)
- পায়ের ছোট জামা (প্যান্ট) বা লুঙ্গি

## এড়িয়ে চলুন:
- ঘন এবং কৃত্রিম ফ্যাব্রিক
- টাইট ফিটিং পোশাক
- গাঢ় রঙ

## অতিরিক্ত টিপস:
- ছাতা বা টুপি ব্যবহার করুন
- হালকা স্কার্ফ রাখুন রৌদ্র থেকে বাঁচতে
- ঘাম শোষক ইনসোল ব্যবহার করুন

এই পোশাকগুলো আপনাকে ঢাকার তীব্র গরমে সুবিধা দেবে।
Tool calls  : []

=== AIMessage with Tool Call ===
Type        : AIMessage
Content     : [{'text': 'আমি আপনার জন্য ঢাকার আবহাওয়া তথ্য জানতে পারি।', 'type': 'text'}, {'id': 'toolu_015TjdQmbHWn3RnWadH9jYiS', 'caller': {'type': 'direct'}, 'input': {'city': 'Dhaka'}, 'name': 'get_weather', 'type': 'tool_use'}]
Tool calls  : [{'name': 'get_we

---
## 5. ToolMessage

**কী হচ্ছে:** Tool execute হওয়ার পর তার result `ToolMessage`-এ মুড়ে conversation history-তে যোগ করা হয়।

**কেন দরকার:** AI tool call করার পর সে result জানে না — result টা `ToolMessage` হিসেবে পাঠালে তবেই AI সেটা পড়তে পারে এবং final answer লিখতে পারে।

**`tool_call_id` কেন লাগে:** AI একসাথে কয়েকটা tool call করতে পারে। `tool_call_id` দিয়ে AI বুঝতে পারে কোন result কোন call-এর জন্য।

**পুরো flow একসাথে:** নিচে manually সব message type একসাথে জুড়ে একটা complete conversation দেখানো হয়েছে।

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

llm_with_tools = llm.bind_tools([get_weather])

# ── Turn 1: Human asks ──
messages = [
    SystemMessage(content="তুমি একজন আবহাওয়া সহকারী। শুধু get_weather tool ব্যবহার করে উত্তর দেবে।"),
    HumanMessage(content="chittagong er আবহাওয়া জানাও।"),
]

print("=== Turn 1: Human asks ===")
for m in messages:
    print(f"  [{type(m).__name__}] {m.content[:60]}")

# ── Turn 2: AI decides to call the tool ──
ai_response = llm_with_tools.invoke(messages)
messages.append(ai_response)

print("\n=== Turn 2: AIMessage (tool call) ===")
print("  Tool calls:", ai_response.tool_calls)

# ── Turn 3: Execute tool, wrap result in ToolMessage ──
for tc in ai_response.tool_calls:
    result = get_weather.invoke(tc["args"])
    tool_msg = ToolMessage(content=result, tool_call_id=tc["id"])
    messages.append(tool_msg)
    print(f"\n=== Turn 3: ToolMessage ===")
    print(f"  tool_call_id : {tool_msg.tool_call_id}")
    print(f"  content      : {tool_msg.content}")

# ── Turn 4: AI reads ToolMessage and writes final answer ──
final = llm_with_tools.invoke(messages)

print("\n=== Turn 4: Final AIMessage (text reply) ===")
print(final.content)

=== Turn 1: Human asks ===
  [SystemMessage] তুমি একজন আবহাওয়া সহকারী। শুধু get_weather tool ব্যবহার করে
  [HumanMessage] chittagong er আবহাওয়া জানাও।

=== Turn 2: AIMessage (tool call) ===
  Tool calls: [{'name': 'get_weather', 'args': {'city': 'Chittagong'}, 'id': 'toolu_014KAewy7oH56UQK8nqZr4uq', 'type': 'tool_call'}]

=== Turn 3: ToolMessage ===
  tool_call_id : toolu_014KAewy7oH56UQK8nqZr4uq
  content      : Weather in Chittagong:
  Temperature : 32°C (feels like 36°C)
  Condition   : Partly Cloudy
  Humidity    : 78%
  Wind Speed  : 14 km/h

=== Turn 4: Final AIMessage (text reply) ===
চট্টগ্রামের আবহাওয়া:

🌡️ **তাপমাত্রা**: ৩২°C (অনুভূত তাপমাত্রা ৩৬°C)
☁️ **আবহাওয়া**: আংশিক মেঘলা
💧 **আর্দ্রতা**: ৭৮%
💨 **বাতাসের গতি**: ১৪ কিমি/ঘণ্টা

আজ চট্টগ্রামে গরম এবং আর্দ্র আবহাওয়া রয়েছে। বাইরে যেতে গেলে হালকা পোশাক পরে যাওয়া ভালো হবে।


: 

---
## সারসংক্ষেপ

| Message | Class | কখন দরকার |
|---|---|---|
| Text Prompt | `str` | একটাই quick call, shortcut |
| SystemMessage | `SystemMessage` | AI-এর role বা behavior set করতে |
| HumanMessage | `HumanMessage` | User-এর প্রতিটি input |
| AIMessage | `AIMessage` | LLM-এর reply — text বা tool call |
| ToolMessage | `ToolMessage` | Tool-এর result AI-কে ফেরত দিতে |

**মনে রাখো:** সব message একসাথে `messages` list-এ রাখলেই LangChain পুরো conversation context বুঝতে পারে।